# Practical 5: Box-Jenkins ARIMA Modeling for Catfish Sales
## Dataset: catfish.xls (Monthly Catfish Sales)

---
### 📘 Beginner's Concept: What is ARIMA?
**ARIMA** stands for **AutoRegressive Integrated Moving Average**:
1. **AR ($p$)**: AutoRegressive order. Uses past values to predict future values (determined via **PACF** plot).
2. **I ($d$)**: Integrated degree of differencing required to make data stationary.
3. **MA ($q$)**: Moving Average order. Uses past forecast errors to improve predictions (determined via **ACF** plot).
---

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
df = pd.read_csv("catfish.xls")
df.head(5)

In [ ]:
df.shape

In [ ]:
# convert Date column to datetime
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date")
df.head(5)

In [ ]:
sns.lineplot(df)
plt.ylabel("Catfish Sales")
plt.title("Catfish Sales Time Series")
plt.show()

In [ ]:
# ADF and KPSS Test on Raw Data
print("ADF Statistic:", adfuller(df["Total"])[0], "p-value:", adfuller(df["Total"])[1])
print("KPSS p-value:", kpss(df["Total"])[1])

In [ ]:
# First Differencing for Stationarity (d=1)
diff_catfish = df["Total"].diff().dropna()
print("Differenced ADF p-value:", adfuller(diff_catfish)[1])
print("Differenced KPSS p-value:", kpss(diff_catfish)[1])

In [ ]:
# ACF and PACF Plots for Order Selection (p and q)
plot_acf(diff_catfish)
plt.show()
plot_pacf(diff_catfish)
plt.show()

In [ ]:
# Train Test Split (80% Train, 20% Test)
train_df = df[:int(df.shape[0]*0.8)]
test_df = df[int(df.shape[0]*0.8):]

In [ ]:
# Fit ARIMA(1, 1, 1) Model
from statsmodels.tsa.arima.model import ARIMA
model_arima = ARIMA(train_df["Total"], order=(1, 1, 1))
model_arima_fit = model_arima.fit()
print(model_arima_fit.summary())

In [ ]:
# Forecast and MAPE Evaluation
from sklearn.metrics import mean_absolute_percentage_error
forecast_arima = model_arima_fit.forecast(len(test_df))
mape = mean_absolute_percentage_error(test_df["Total"], forecast_arima)
print("ARIMA Forecast MAPE:", mape)

In [ ]:
# Plot Actual vs Forecast
plt.plot(df["Total"], label="Original Data")
plt.plot(forecast_arima, label="ARIMA Forecast")
plt.legend()
plt.title("Catfish Sales ARIMA Forecast")
plt.show()